## 1. Import Required Libraries

In [1]:
import os
import re
import joblib
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


## 2. Create Training Dataset



In [3]:
data = [
    # Scam / phishing examples
    ("Urgent! Your bank account has been locked. Verify your identity immediately at http://secure-bank-login.example.com", 1),
    ("Your parcel delivery failed. Pay a small redelivery fee now at bit.ly/claim-package", 1),
    ("Congratulations! You have won a lottery prize. Send your bank details to claim now", 1),
    ("Your card will be permanently suspended within 2 hours unless you confirm your OTP", 1),
    ("Work from home job selected. Pay refundable registration fee to confirm your seat", 1),
    ("Security alert: unauthorized login detected. Click this link to reset your password", 1),
    ("Dear customer, update KYC immediately or your account will be blocked today", 1),
    ("You are eligible for instant loan approval. Send Aadhaar and bank details on WhatsApp", 1),
    ("Final notice: tax refund pending. Enter card number and CVV to receive payment", 1),
    ("Claim your free gift card now. Limited time offer, click the link", 1),
    ("Your PayPal account is restricted. Login here to avoid permanent suspension", 1),
    ("Police verification required. Pay penalty immediately to avoid legal action", 1),
    ("Your electricity bill is unpaid. Service will be disconnected tonight. Pay now", 1),
    ("You have been selected for a government subsidy. Send OTP to receive funds", 1),
    ("Netflix payment failed. Update billing information using this urgent link", 1),

    # Legitimate / safe examples
    ("Hi, are we still meeting at 5 pm today?", 0),
    ("Your order has been delivered. Thank you for shopping with us", 0),
    ("Reminder: your appointment is scheduled for Monday at 10 AM", 0),
    ("Can you send me the project report when you get time?", 0),
    ("Team meeting starts in 15 minutes. Please join the call", 0),
    ("Your monthly statement is ready in the official banking app", 0),
    ("Happy birthday! Hope you have a great day", 0),
    ("The package is out for delivery and will arrive by evening", 0),
    ("Please review the attached notes before tomorrow's class", 0),
    ("Your password was changed successfully from your account settings", 0),
    ("Lunch at 1? Let me know if that works", 0),
    ("Your ticket has been booked. Check the official app for details", 0),
    ("Payment received. Invoice has been sent to your registered email", 0),
    ("Your doctor consultation is confirmed for Friday", 0),
    ("Please submit the assignment before the deadline", 0),
]

df = pd.DataFrame(data, columns=["message", "label"])
df.head()


,message,label
0,Urgent! Your bank account has been locked. Ver...,1
1,Your parcel delivery failed. Pay a small redel...,1
2,Congratulations! You have won a lottery prize....,1
3,Your card will be permanently suspended within...,1
4,Work from home job selected. Pay refundable re...,1


## 3. Text PreProcessing

In [4]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " urltoken ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_message"] = df["message"].apply(clean_text)
df[["message", "clean_message", "label"]].head()


,message,clean_message,label
0,Urgent! Your bank account has been locked. Ver...,urgent your bank account has been locked verif...,1
1,Your parcel delivery failed. Pay a small redel...,your parcel delivery failed pay a small redeli...,1
2,Congratulations! You have won a lottery prize....,congratulations you have won a lottery prize s...,1
3,Your card will be permanently suspended within...,your card will be permanently suspended within...,1
4,Work from home job selected. Pay refundable re...,work from home job selected pay refundable reg...,1


## 4. Split Dataset

In [5]:
X = df["clean_message"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


Training samples: 22
Testing samples: 8


## 5.  ML Pipeline



In [6]:
model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1)),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

model.fit(X_train, y_train)
print("Model training complete")


Model training complete


## 6. Evaluate Model

In [7]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=["Legitimate", "Scam"]))


Accuracy: 0.5

Confusion Matrix:
 [[1 3]
 [1 3]]

Classification Report:
               precision    recall  f1-score   support

  Legitimate       0.50      0.25      0.33         4
        Scam       0.50      0.75      0.60         4

    accuracy                           0.50         8
   macro avg       0.50      0.50      0.47         8
weighted avg       0.50      0.50      0.47         8



## 7. Add Fraud Signal Helpers


In [8]:
SUSPICIOUS_PHRASES = [
    "urgent", "immediately", "act now", "limited time", "last chance",
    "account locked", "account has been locked", "suspended", "blocked",
    "verify your identity", "otp", "password", "pin", "cvv",
    "bank details", "aadhaar", "card number", "registration fee",
    "refundable", "you have won", "lottery", "claim now",
    "free gift", "work from home", "selected"
]

def detect_urls(text):
    return re.findall(r"\b(?:https?://)?(?:www\.)?[a-zA-Z0-9-]+(?:\.[a-zA-Z0-9-]+)+(?:/[^\s]*)?", text)

def find_suspicious_phrases(text):
    lowered = text.lower()
    return [phrase for phrase in SUSPICIOUS_PHRASES if phrase in lowered]

def url_risk_reasons(url):
    reasons = []
    lowered = url.lower()
    if not lowered.startswith("https://"):
        reasons.append("URL is not explicitly HTTPS")
    if any(shortener in lowered for shortener in ["bit.ly", "tinyurl", "t.co"]):
        reasons.append("Shortened URL hides the real destination")
    if re.search(r"\d", lowered.split(".")[0]):
        reasons.append("Domain contains numbers")
    return reasons


## 8. Prediction Function for Backend API



In [9]:
def predict_scam(message):
    cleaned = clean_text(message)
    probability = model.predict_proba([cleaned])[0][1]
    scam_score = round(probability * 100, 2)
    prediction = "Scam" if probability >= 0.5 else "Legitimate"

    urls = detect_urls(message)
    suspicious_phrases = find_suspicious_phrases(message)
    url_analysis = [{"url": url, "reasons": url_risk_reasons(url)} for url in urls]

    if scam_score >= 75:
        risk_level = "High"
    elif scam_score >= 45:
        risk_level = "Medium"
    else:
        risk_level = "Low"

    return {
        "prediction": prediction,
        "scam_score": scam_score,
        "risk_level": risk_level,
        "suspicious_phrases": suspicious_phrases,
        "urls_detected": urls,
        "url_analysis": url_analysis,
        "explanation": "Prediction is based on ML probability plus detected scam-like wording and URL signals."
    }


## 9. Test Prediction

In [10]:
test_message = "Urgent! Your account is blocked. Verify OTP now at http://fake-bank-login.com"
predict_scam(test_message)


{'prediction': 'Scam',
 'scam_score': 56.97,
 'risk_level': 'Medium',
 'suspicious_phrases': ['urgent', 'blocked', 'otp'],
 'urls_detected': ['http://fake-bank-login.com'],
 'url_analysis': [{'url': 'http://fake-bank-login.com',
   'reasons': ['URL is not explicitly HTTPS']}],
 'explanation': 'Prediction is based on ML probability plus detected scam-like wording and URL signals.'}

## 10. Save Model as'pkl'



In [11]:
os.makedirs("models", exist_ok=True)

joblib.dump(model, "models/scam_detector_model.pkl")
print("Saved model to models/scam_detector_model.pkl")


Saved model to models/scam_detector_model.pkl


## 11. LLM Integration

In [ ]:
import json
import urllib.request

def groq_llm_analysis(message, model_name="llama-3.3-70b-versatile"):
    api_key = os.getenv("GROQ_API_KEY")
    if not api_key:
        raise ValueError("Set GROQ_API_KEY before calling GroqCloud.")

    payload = {
        "model": model_name,
        "stream": False,
        "temperature": 0.1,
        "messages": [
            {
                "role": "system",
                "content": "You are a cyber safety assistant. Return only valid JSON."
            },
            {
                "role": "user",
                "content": (
                    "Analyze this message for scam risk. Return JSON with keys: "
                    "prediction, scam_score, risk_level, suspicious_phrases, "
                    "urls_detected, url_analysis, fraud_patterns, explanation.\n\n"
                    f"Message:\n{message}"
                )
            }
        ]
    }

    req = urllib.request.Request(
        "https://api.groq.com/openai/v1/chat/completions",
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {api_key}"
        },
        method="POST"
    )

    with urllib.request.urlopen(req, timeout=45) as response:
        data = json.loads(response.read().decode("utf-8"))

    return json.loads(data["choices"][0]["message"]["content"])




In [ ]:
# Run this cell from the project root to retrain the model with the real datasets.
# It creates/updates models/scam_detector_model.pkl and models/training_metrics.json.

!python train_from_datasets.py


In [ ]:
import json

with open('models/training_metrics.json', 'r', encoding='utf-8') as metrics_file:
    metrics = json.load(metrics_file)

print('Rows used:', metrics['dataset_rows'])
print('Legitimate:', metrics['class_counts']['legitimate'])
print('Scam/Spam:', metrics['class_counts']['scam_or_spam'])
print('Accuracy:', round(metrics['accuracy'] * 100, 2), '%')
print('Confusion matrix:', metrics['confusion_matrix'])
